# CP1 · Examen blanc de la partie I — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/checkpoints/partie_1/03_examen_solutions.ipynb)

L'enregistrement des réponses que vérifie le notebook de l'examen (cellules marquées `answer`, récoltées par `tools/build_answers.py`), calculées à partir des nombres du sujet ; les erreurs classiques reçoivent un message qui dit où chercher. Le corrigé détaillé et le barème sont dans `03_examen_corrige.md`.

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

🧪 workbookMaths : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | scipy 1.16.3 | sympy 1.14.0 | matplotlib 3.10.0 | pandas 2.2.3 | scikit-learn 1.6.1
   Graine : 42 | FAST_MODE : True


## Les réponses vérifiées après l'examen

In [2]:
# The answers are computed from the numbers of the statements (01_examen_sujet.md)
import math
from fractions import Fraction


def H(*probs):
    """Entropy in bits of a law given by its probabilities (0 log 0 counts as 0)."""
    return -sum(p * math.log2(p) for p in probs if p > 0)


# CP1.2: the binomial B(12, 0.08), the Poisson law of 1.8 alerts per hour, and the table values of Φ
P_2 = 0.07
PHI_1, PHI_2 = 0.8413, 0.9772

# CP1.3: the joint law of (X, Y), X = 0, 1, 2 (rows), Y = 0, 1 (columns)
JOINT_3 = {(0, 0): 0.1, (0, 1): 0.2, (1, 0): 0.2, (1, 1): 0.1, (2, 0): 0.1, (2, 1): 0.3}
E_X3 = sum(x * p for (x, y), p in JOINT_3.items())
E_Y3 = sum(y * p for (x, y), p in JOINT_3.items())
V_X3 = sum(x * x * p for (x, y), p in JOINT_3.items()) - E_X3 ** 2
V_Y3 = sum(y * y * p for (x, y), p in JOINT_3.items()) - E_Y3 ** 2
COV_3 = sum(x * y * p for (x, y), p in JOINT_3.items()) - E_X3 * E_Y3

# CP1.4: 50 lengths, their sum and the sum of the squared deviations
N_4, SUM_4, SS_4 = 50, 1230, 1911
MEAN_4 = SUM_4 / N_4
S_4 = math.sqrt(SS_4 / (N_4 - 1))                 # ddof = 1
SE_4 = S_4 / math.sqrt(N_4)
P_4 = 18 / 50
SE_P4 = math.sqrt(P_4 * (1 - P_4) / N_4)

# CP1.6: the prior Beta(4, 8), then 7 acceptances out of 30, then 3 out of 10
A_6, B_6 = 4, 8

# CP1.7: Newton on x³ + x − 3 from 1, and a log-sum-exp of very negative scores
NEWTON_7 = [1.0]
for _ in range(2):
    x = NEWTON_7[-1]
    NEWTON_7.append(x - (x ** 3 + x - 3) / (3 * x ** 2 + 1))
SCORES_7 = [-790, -787, -785]
LSE_7 = max(SCORES_7) + math.log(sum(math.exp(s - max(SCORES_7)) for s in SCORES_7))

# CP1.8: f(x, y) = x³ − 6xy + 3y², at the point (1, −1)
GRAD_8 = [3 * 1 ** 2 - 6 * (-1), -6 * 1 + 6 * (-1)]

# CP1.9: the frequencies p, the old model q, and the joint law of (F, Y)
P_9, Q_9 = (0.45, 0.4, 0.15), (0.6, 0.35, 0.05)
CROSS_9 = -sum(p * math.log2(q) for p, q in zip(P_9, Q_9))
H_Y_F9 = 0.4 * H(0.35 / 0.4, 0.05 / 0.4) + 0.6 * H(0.10 / 0.6, 0.50 / 0.6)

# CP1.12: Newton on g(lam) = lam − 1.6 (1 − e^(−lam)) from 1.6, then the root
XBAR_12 = 32 / 20
NEWTON_12 = [1.6]
for _ in range(30):
    lam = NEWTON_12[-1]
    NEWTON_12.append(lam - (lam - XBAR_12 * (1 - math.exp(-lam))) / (1 - XBAR_12 * math.exp(-lam)))
LAM_12 = NEWTON_12[-1]

**CP1.1 — Intégrales et densité : une aire, une fonction de répartition, un quantile**

In [3]:
wbm.record("CP1.1a", 4.0, decimals=3, mistakes={"par parties : ∫u v' = [u v] − ∫u' v, avec le signe moins": 8 * math.e - 4, "une primitive de e^(x/2) n'est pas e^(x/2) : dérive ta primitive pour vérifier": math.e + 1, "évalue aussi chaque crochet en 0 : e⁰ = 1": 0, "l'intégration par parties laisse une seconde intégrale à calculer": 4 * math.e})
wbm.record("CP1.1b", Fraction(1, 4), kind="fraction", simplified=True, mistakes={"intègre les deux termes de x + 1 sur [0 ; 2]": "1/2", "c'est l'aire sous x + 1 ; l'aire sous f = c (x + 1) doit valoir 1": "4"})
wbm.record("CP1.1c", "(x**2 + 2*x)/8", kind="expr", variables=("x",), mistakes={"la densité est c (x + 1), pas x + 1 : F(2) doit valoir 1": "x**2/2 + x", "F est la primitive de f qui vaut 0 en 0, pas f elle-même": "(x + 1)/4", "F(0) doit valoir 0 : choisis la primitive qui s'annule en 0": "(x + 1)**2/8"})
wbm.record("CP1.1d", Fraction(1) - Fraction(3, 8), kind="fraction", simplified=True, mistakes={"c'est P(X ≤ 1) ; la question demande la probabilité de l'intervalle ]1 ; 2]": "3/8", "c'est une différence de valeurs de la densité, pas de la fonction de répartition": "1/4"})
wbm.record("CP1.1e", -1 + math.sqrt(5), decimals=3, mistakes={"la médiane vérifie F(m) = 1/2 ; ce n'est pas l'espérance": 7 / 6, "une médiane est une valeur que X peut prendre : dans quel intervalle doit-elle tomber ?": -1 - math.sqrt(5), "la densité n'est pas constante : la médiane n'est pas le milieu de l'intervalle": 1, "résous F(m) = 1/2 avec la fonction de répartition de c), constante comprise": -1 + math.sqrt(2)})
wbm.record("CP1.1f", Fraction(1, 4) * (Fraction(8, 3) + Fraction(2)), kind="fraction", simplified=True, mistakes={"E[X] = ∫ x f(x) dx : la densité n'est pas constante": "1", "la densité est c (x + 1), pas x + 1": "14/3"})

WBM_ANSWER {"coarse_any": {"0": ["d2215451685d172d8d03a3b7b627100c4bf5f619146b439206f7d8d08ecc7c0f"]}, "coarser": {"1": ["2e259616f1f83adf502f5e8f61dd7d3944306cced6c4249b6a98660b5d3569f2"]}, "decimals": 3, "hash": "f6ac76b40d1346ea75b6e5755f7829188546f46337d1358cfa8206012abb4c0f", "hash_coarse": "120440461dbfa8672753c18181bc32185c421f20748a54a2801b7ea5f9c5c874", "id": "CP1.1a", "kind": "float", "magnitude": 0, "mistakes": {"135cd31f5e526249f2baa3010394a92e70049d2791ac6d1c4d9b169745926680": "l'intégration par parties laisse une seconde intégrale à calculer", "64d1dff2cf44db59be9045d1774fd76af217b4facf052aa7a7c2e2b889741910": "une primitive de e^(x/2) n'est pas e^(x/2) : dérive ta primitive pour vérifier", "75c1466e93e664d34f3ddb7af112235d667bb18db742ae30d523aba6731124a5": "évalue aussi chaque crochet en 0 : e⁰ = 1", "aa7f0d27f74db923e1029cb476cc51e0496a2010a919ea8296c5fced3fec5852": "par parties : ∫u v' = [u v] − ∫u' v, avec le signe moins"}, "sign": 1}
📝 Ex CP1.1a : réponse enregistrée

WBM_ANSWER {"hash": "467ef6d09b51c9c4222f620607a86ea3e633b5cb4ee9d6183e1c58f2089013a7", "id": "CP1.1b", "kind": "fraction", "mistakes": {"93c1269ac44f6d0ebbd6824b90b41f35c17730b451e66f1d1dbb14a1c7f20e66": "c'est l'aire sous x + 1 ; l'aire sous f = c (x + 1) doit valoir 1", "b3514980388518da480f5ff44b9f3a758b0cc9117e81ae1d463d8d1a0af55133": "intègre les deux termes de x + 1 sur [0 ; 2]"}, "sign": 1, "simplified": true}
📝 Ex CP1.1b : réponse enregistrée (fraction).
WBM_ANSWER {"hash": "0c4de926d2307e28dc4313997c2dfe2047a8730d81b314b0b8259429f9dd0722", "id": "CP1.1c", "kind": "expr", "mistakes": {"1412e1985c4269fb456afa99e1ebdf5d1ecf2de9437f0a70d7600dabe7cb159f": "F est la primitive de f qui vaut 0 en 0, pas f elle-même", "2af3fb0a663ceabd0dbfe6aaa7931db90ed8c2c787c6abba163375ba68756e59": "F(0) doit valoir 0 : choisis la primitive qui s'annule en 0", "60341bedb35e154def5e5d411e96645a26b3e734639b9bab607aae40287cd242": "la densité est c (x + 1), pas x + 1 : F(2) doit valoir 1"}, "points": [

**CP1.2 — Lois usuelles : binomiale, Poisson, normale**

In [4]:
wbm.record("CP1.2a", (1 - P_2) ** 12, decimals=4, mistakes={"aucune erreur : chaque case doit être lue juste": P_2 ** 12})
wbm.record("CP1.2b", 1 - (1 - P_2) ** 12 - 12 * P_2 * (1 - P_2) ** 11, decimals=4, mistakes={"« au moins deux » exclut aussi le cas d'une seule erreur": 1 - (1 - P_2) ** 12, "l'erreur unique peut être sur n'importe laquelle des 12 cases : le coefficient binomial": 1 - (1 - P_2) ** 12 - P_2 * (1 - P_2) ** 11, "« au moins deux » compte aussi 3, 4… erreurs : passe par l'événement contraire": 66 * P_2 ** 2 * (1 - P_2) ** 10})
wbm.record("CP1.2c", math.exp(-1.8) * 1.8 ** 2 / 2, decimals=4, mistakes={"la loi de Poisson divise par k!": math.exp(-1.8) * 1.8 ** 2, "« exactement 2 » ne compte pas les heures à 0 ou 1 alerte": math.exp(-1.8) * (1 + 1.8 + 1.8 ** 2 / 2)})
wbm.record("CP1.2d", 1 - math.exp(-1.8 / 3), decimals=4, mistakes={"le paramètre 1,8 est celui d'une heure : quel est le nombre moyen d'alertes sur une autre durée ?": 1 - math.exp(-1.8), "le paramètre est proportionnel à la durée : quelle fraction d'heure font 20 minutes ?": 1 - math.exp(-1.8 / 20), "« au moins une » compte aussi 2, 3… alertes": 0.6 * math.exp(-0.6)})
wbm.record("CP1.2e", PHI_2 - (1 - PHI_1), decimals=2, mistakes={"12,5 minutes est sous la moyenne : son z est négatif": PHI_2 - PHI_1, "c'est la probabilité d'être en dehors de l'intervalle": 1 - (PHI_2 - (1 - PHI_1))})
wbm.record("CP1.2f", 14 - 1.2816 * 1.5, decimals=2, mistakes={"dépassée par 90 % des batteries : seules 10 % font moins, la valeur est sous la moyenne": 14 + 1.2816 * 1.5, "90 % au-dessus laisse 10 % en dessous : cherche z tel que Φ(z) = 0,10": 14 - 1.645 * 1.5, "z s'exprime en écarts-types : repasse en minutes": 14 - 1.2816})

WBM_ANSWER {"coarse_any": {"1": ["553225cefc8ad3560361b7d9d50c0ba2889731b92f6de11bc7f7961cda6efca1"]}, "coarser": {"2": ["564bf3804468f7a76fce9438c9892be971c1e25ba5b234ca78d8e4c7b82958e1"]}, "decimals": 4, "hash": "2f0a89668dce060b4dae24ed26fee46e48fe05dc153bb20aedab0fa529684227", "hash_coarse": "ea068d971fed932fb45509fba25585e394583f67522a5926036981e0969a0b7f", "id": "CP1.2a", "kind": "float", "magnitude": -1, "mistakes": {"bd69c8239c8b3d9db751f19333ca4be39ba241ede694761c4c72a1a60310218a": "aucune erreur : chaque case doit être lue juste"}, "sign": 1}
📝 Ex CP1.2a : réponse enregistrée (float, 4 décimales).
WBM_ANSWER {"coarse_any": {"1": ["28a22adc1bd74fd71359f68d1a4315ff970e54ac2de44c6394cbd1fe2828691f"]}, "coarser": {"2": ["4a0d7fba6870d7ed02434aa5b8ece5440a7d3cb06db6efc4a5bc347c71b12518"]}, "decimals": 4, "hash": "e6b2a2a5fd9f8caa5261ed8f1aaa148e37673ac06a72df8009dd0bd7c51ccd5c", "hash_coarse": "378d8991126cfb79801253e8514a7b9e755101f9b26bc362f5c4d00f00c58432", "id": "CP1.2b", "kin

**CP1.3 — Un couple de variables : marginales, indépendance, variance d'une somme**

In [5]:
wbm.record("CP1.3a", [0.3, 0.3, 0.4], decimals=2, mistakes={"la loi de X additionne les deux cases de chaque ligne": [0.1, 0.2, 0.1]})
wbm.record("CP1.3b", False, mistakes={"compare chaque case au produit de ses deux marginales": True})
wbm.record("CP1.3c", COV_3, decimals=2, mistakes={"c'est E[XY] ; la covariance compare E[XY] à ce que donneraient des variables indépendantes": COV_3 + E_X3 * E_Y3, "X et Y ne sont pas indépendantes : calcule E[XY] avec la loi jointe": 0})
wbm.record("CP1.3d", V_X3, decimals=2, mistakes={"c'est E[X²] ; la variance mesure la dispersion autour de l'espérance": V_X3 + E_X3 ** 2, "c'est l'espérance ; la variance mesure l'écart à l'espérance": E_X3, "la variance retranche le carré de l'espérance, pas l'espérance": V_X3 + E_X3 ** 2 - E_X3})
wbm.record("CP1.3e", V_X3 + V_Y3 + 2 * COV_3, decimals=2, mistakes={"X et Y ne sont pas indépendantes : la variance d'une somme ajoute 2 Cov(X, Y)": V_X3 + V_Y3, "relis la formule de la variance d'une somme : combien de fois la covariance y apparaît-elle ?": V_X3 + V_Y3 + COV_3, "la variance de Y n'est pas E[Y²] : Y vaut 0 ou 1, c'est une Bernoulli": V_X3 + E_Y3 + 2 * COV_3})

WBM_ANSWER {"decimals": 2, "element_coarse": [["0bf2e44fc179b08acca528e9bf9d78d4838bf9ac83875a7c1439bb2eb855ce97"], ["3b55e1d0350299d33faf19b753caed6246787f9bf50bcc0b3ace75d42a835af1"], ["2027b95cb06a7441b92b8cd2dfcd5750c68aedb5e401e5ca05fb93ea6e0b252a"]], "element_coarse_any": [[], [], []], "element_hashes": ["00376ae70da3247bd5bb4201d79df32b07f5c15aad64d7ca37acd3d86da0e845", "91fdb82bf311d10e80065f1a40cd5a9c5912eef42e58f165aa9835f96c002612", "6412e14f04dda1da6b8c9799f1572a6320c0d814b87a91b49ab4e18fb66638c2"], "hash": "e838409fb674efb5a868d4641a2e47a861ef1e21da30efc696771d060e214a5d", "id": "CP1.3a", "kind": "array", "mistakes": {"6d1a52694e190323f6bf595a716cfd833d0e06adabd51f335e5b0a43d56e1f50": "la loi de X additionne les deux cases de chaque ligne"}, "shape": [3]}
📝 Ex CP1.3a : réponse enregistrée (array, 2 décimales).
WBM_ANSWER {"hash": "67ef418b325576e4a100619559b7a11fe89d2194d6397583ee75ab21b8032ba4", "id": "CP1.3b", "kind": "bool", "mistakes": {"e2c964dae47d04771be7404084a320d

**CP1.4 — Estimer une moyenne : erreur type et intervalle de confiance**

In [6]:
wbm.record("CP1.4a", MEAN_4, decimals=1, mistakes={"la moyenne divise la somme par n, pas par n − 1": SUM_4 / (N_4 - 1)})
wbm.record("CP1.4b", S_4, decimals=3, mistakes={"ddof = 1 : divise la somme des carrés des écarts par n − 1": math.sqrt(SS_4 / N_4), "c'est la variance ; la question demande l'écart-type": SS_4 / (N_4 - 1)})
wbm.record("CP1.4c", SE_4, decimals=3, mistakes={"l'erreur type divise l'écart-type par √n, pas par n": S_4 / N_4, "l'erreur type part de l'écart-type s, pas de la variance": SS_4 / (N_4 - 1) / N_4, "prends l'écart-type avec ddof = 1": math.sqrt(SS_4 / N_4) / math.sqrt(N_4), "l'erreur type divise par √n, avec n = 50 mesures": S_4 / math.sqrt(N_4 - 1)})
wbm.record("CP1.4d", [MEAN_4 - 1.96 * SE_4, MEAN_4 + 1.96 * SE_4], decimals=2, mistakes={"l'intervalle de la moyenne utilise l'erreur type, pas l'écart-type des données": [MEAN_4 - 1.96 * S_4, MEAN_4 + 1.96 * S_4], "à 95 %, le quantile est 1,96": [MEAN_4 - 1.645 * SE_4, MEAN_4 + 1.645 * SE_4], "un intervalle à 95 % s'étend de combien d'erreurs types de part et d'autre ?": [MEAN_4 - SE_4, MEAN_4 + SE_4]})
wbm.record("CP1.4e", 1.645 * SE_4, decimals=2, mistakes={"l'intervalle de la moyenne utilise l'erreur type, pas l'écart-type des données": 1.645 * S_4, "à 90 %, le quantile est 1,645": 1.96 * SE_4, "la demi-largeur est la moitié de la largeur de l'intervalle": 2 * 1.645 * SE_4})
wbm.record("CP1.4f", [P_4 - 1.96 * SE_P4, P_4 + 1.96 * SE_P4], decimals=3, mistakes={"l'erreur type d'une proportion est la racine de p̂ (1 − p̂) / n": [P_4 - 1.96 * P_4 * (1 - P_4) / N_4, P_4 + 1.96 * P_4 * (1 - P_4) / N_4], "l'erreur type d'une proportion divise par n, pas par n − 1": [P_4 - 1.96 * math.sqrt(P_4 * (1 - P_4) / (N_4 - 1)), P_4 + 1.96 * math.sqrt(P_4 * (1 - P_4) / (N_4 - 1))]})

WBM_ANSWER {"decimals": 1, "hash": "12411b07b4b06f8e8c54b4fc76f31becdf8fecf9dbfa30474367e02d2fdeb191", "hash_coarse": "c016c1fa6aabaed70726787155a6a011df92e1d8fffb76edd48277caf6080af1", "id": "CP1.4a", "kind": "float", "magnitude": 1, "mistakes": {"ddbb1d45989cb96e67c58bdc5622e0c3687038e90ee167fe068f7d405740e4fa": "la moyenne divise la somme par n, pas par n − 1"}, "sign": 1}
📝 Ex CP1.4a : réponse enregistrée (float, 1 décimale).
WBM_ANSWER {"coarse_any": {"0": ["5eabfb02ff9294d32bb56e69764efdf7740ad0b5191d0c1129bce16127eaf864"]}, "coarser": {"1": ["a9ccf9eaee10ef4d7b31c2c98127693ad2506f40c9ffa392e582d7360ce59139"]}, "decimals": 3, "hash": "b2a767e7008bfb36a901c67ec09279e7a4223bc7f8e5cd34c47ac8712f792b13", "hash_coarse": "1dce54841fc4edaa436834f895cbddad570589bc8971ef3a4655d54fbf6ad847", "id": "CP1.4b", "kind": "float", "magnitude": 0, "mistakes": {"4aa949a03339e6e16cc3b7088d286f0d6426f5053037ccd3660102a86eaaaa7d": "c'est la variance ; la question demande l'écart-type", "c654388e76e5cf

**CP1.5 — Un tableau de contingence et des métriques comme probabilités**

In [7]:
wbm.record("CP1.5a", Fraction(45 + 30, 500), kind="fraction", simplified=True, mistakes={"compte toutes les alertes, vraies et fausses": "9/100", "c'est la part des vraies chutes ; la question demande la part des alertes": "3/25"})
wbm.record("CP1.5b", Fraction(45, 45 + 30), kind="fraction", simplified=True, mistakes={"la precision conditionne par l'alerte : parmi les alertes, la part de vraies chutes": "3/4"})
wbm.record("CP1.5c", Fraction(45, 60), kind="fraction", simplified=True, mistakes={"le recall conditionne par la vérité : parmi les vraies chutes, la part qui a donné l'alerte": "3/5"})
wbm.record("CP1.5d", 2 * Fraction(3, 5) * Fraction(3, 4) / (Fraction(3, 5) + Fraction(3, 4)), kind="fraction", simplified=True, mistakes={"le F1 est la moyenne harmonique de la precision et du recall, pas leur moyenne arithmétique": "27/40", "relis la formule du F1 : il manque une partie de la fraction": "9/10", "le produit de la precision et du recall n'est pas le F1 : relis la formule de la moyenne harmonique": "9/20"})
wbm.record("CP1.5e", 410 / 440, decimals=4, mistakes={"la spécificité conditionne par la vérité (pas de chute), pas par la réponse de la montre (pas d'alerte)": 410 / 425, "la spécificité est une probabilité conditionnelle : parmi quels événements ?": 410 / 500})
wbm.record("CP1.5f", False, mistakes={"compare P(alerte et chute) au produit P(alerte) × P(chute)": True})

WBM_ANSWER {"hash": "26caff0426f213fda19d7a28511fe1bf43d7286f87d2ff70c991034298713a85", "id": "CP1.5a", "kind": "fraction", "mistakes": {"27a07922c9b794bb01d30e3efef3deb87b8e7a296a6b2a03af180503acaa8847": "c'est la part des vraies chutes ; la question demande la part des alertes", "5f324467f9073034066bf6b09a6afe2c59068a886d1472f0c9877a956e8b35fa": "compte toutes les alertes, vraies et fausses"}, "sign": 1, "simplified": true}
📝 Ex CP1.5a : réponse enregistrée (fraction).
WBM_ANSWER {"hash": "2931bfffb41f1a12526c6d237039145d850383cb0ddb862ba75b0ab478be6fc4", "id": "CP1.5b", "kind": "fraction", "mistakes": {"247ca9a6e8c9d8ed69ceb625a1c3c56133b7ff990b880be795d0419c3ae819cf": "la precision conditionne par l'alerte : parmi les alertes, la part de vraies chutes"}, "sign": 1, "simplified": true}
📝 Ex CP1.5b : réponse enregistrée (fraction).
WBM_ANSWER {"hash": "ea49282eb23a8215b2ef858c0178c6af858133b37bd0bca02d7509d776607310", "id": "CP1.5c", "kind": "fraction", "mistakes": {"2d61c25b65cae448

**CP1.6 — Bayes et la loi Beta : une mise à jour conjuguée**

In [8]:
wbm.record("CP1.6a", Fraction(A_6, A_6 + B_6), kind="fraction", simplified=True, mistakes={"la moyenne de Beta(a, b) n'est pas a/b : relis sa formule (M04 §4)": "1/2", "c'est le mode de la loi ; la question demande sa moyenne": "3/10"})
wbm.record("CP1.6b", [A_6 + 7, B_6 + 23], mistakes={"b reçoit le nombre d'échecs, pas le nombre d'essais": [11, 38], "a reçoit les succès, b les échecs": [27, 15], "la liste demandée est [a, b], dans cet ordre": [31, 11], "ajoute les succès et les échecs aux paramètres de l'a priori": [7, 23]})
wbm.record("CP1.6c", Fraction(A_6 + 7, A_6 + B_6 + 30), kind="fraction", simplified=True, mistakes={"c'est la fréquence observée ; la moyenne a posteriori tient compte aussi de l'a priori": "7/30", "c'est le mode ; la question demande la moyenne": "1/4"})
wbm.record("CP1.6d", Fraction(A_6 + 7 - 1, A_6 + B_6 + 30 - 2), kind="fraction", simplified=True, mistakes={"c'est la moyenne ; le MAP est le mode de la loi a posteriori": "11/42", "le MAP tient compte de l'a priori : c'est le mode de la loi a posteriori": "7/30"})
wbm.record("CP1.6e", Fraction(A_6 + 10, A_6 + B_6 + 40), kind="fraction", simplified=True, mistakes={"l'a priori de la deuxième mise à jour est l'a posteriori de la première": "7/22", "ajoute aussi les 10 nouvelles propositions": "11/42", "b reçoit le nombre d'échecs, pas le nombre d'essais": "14/55"})
wbm.record("CP1.6f", 14 / 52 * 15 / 53, decimals=4, mistakes={"la deuxième acceptation se calcule avec la loi mise à jour par la première, pas avec la même moyenne": (14 / 52) ** 2, "la question demande que les deux soient acceptées": 14 / 52})

WBM_ANSWER {"hash": "7f0e2b8b3e3ad6a5d8117605ed5f3587412c38716f062582aea6d4ae7f9b5d30", "id": "CP1.6a", "kind": "fraction", "mistakes": {"4a9c4c3496a026c02ef9f1fedc9d062206f319d3417c7fb15062b7d0807fe320": "c'est le mode de la loi ; la question demande sa moyenne", "6a425f33fe7f65a1c8f7d3a6f42e6ba8e86d8f05713fddfaf27e6eb9d1e473e9": "la moyenne de Beta(a, b) n'est pas a/b : relis sa formule (M04 §4)"}, "sign": 1, "simplified": true}
📝 Ex CP1.6a : réponse enregistrée (fraction).
WBM_ANSWER {"decimals": 0, "element_hashes": ["746738b0c9854efe3dbf13ed653b033fd3df5b0944ce0d41a76ea68e1de0d9a8", "e6504e97dbbeea06a9517223ff09e766670f82e4f3abc97fa6576ab4f579c957"], "hash": "46cec59ed9ab059c8d2d5d4430f362f31e84592e603f3686a9d02acccdf2d84a", "id": "CP1.6b", "integer": true, "kind": "array", "mistakes": {"210a2ce22296f0cf0db12a43115ff516808d0128bf0e8df7bc59dbf27a159c8c": "a reçoit les succès, b les échecs", "36f3d2e708996b193f69f007dfd9f9f62aa793f30255b8fc2f334a23768342d4": "b reçoit le nombre d'éc

**CP1.7 — Taylor, Newton et un calcul stable**

In [9]:
wbm.record("CP1.7a", "2 + x/4 - x**2/64", kind="expr", variables=("x",), mistakes={"le terme d'ordre 2 est f''(0) x² / 2 : n'oublie pas le 2": "2 + x/4 - x**2/32", "ce sont les coefficients de √(1 + x) : ici la racine porte sur 4 + x": "2 + x/2 - x**2/8", "l'ordre 2 ajoute un terme en x²": "2 + x/4", "la dérivée seconde de √(4 + x) est négative": "2 + x/4 + x**2/64"})
wbm.record("CP1.7b", 2 + 0.4 / 4 - 0.4 ** 2 / 64, decimals=4, mistakes={"l'ordre 2 ajoute un terme en x²": 2.1, "le terme d'ordre 2 est f''(0) x² / 2 : n'oublie pas le 2": 2 + 0.4 / 4 - 0.4 ** 2 / 32, "c'est la valeur exacte ; la question demande celle que donne le polynôme": math.sqrt(4.4)})
wbm.record("CP1.7c", NEWTON_7[1], decimals=2, mistakes={"le pas de Newton est f(x)/f'(x), pas l'inverse": 1 - 4 / (-1), "x₁ = x₀ − f(x₀)/f'(x₀) : attention au signe de f(1)": 0.75})
wbm.record("CP1.7d", NEWTON_7[2], decimals=4, mistakes={"c'est (presque) la racine ; la question demande le deuxième pas, seulement": 1.213411662762183})
wbm.record("CP1.7e", LSE_7, decimals=3, mistakes={"le log-sum-exp est un peu plus grand que le plus grand des scores": -785, "le terme du plus grand score compte aussi : e⁰ = 1 dans la somme des écarts": -785 + math.log(math.exp(-2) + math.exp(-5))})
wbm.record("CP1.7f", math.exp(-785 - LSE_7), decimals=3, mistakes={"c'est la probabilité d'un autre score : relis lequel est −785": math.exp(-787 - LSE_7), "la softmax prend l'exponentielle des scores avant de normaliser": 785 / (790 + 787 + 785)})

WBM_ANSWER {"hash": "08750caf423d6003298837db6ea55fbf1346203324aea67b47a85f27dcef85cb", "id": "CP1.7a", "kind": "expr", "mistakes": {"4405c5e8a68358f3525a01e0d9c2f8ad9419bf31ac86d22da8b3389b56e85315": "ce sont les coefficients de √(1 + x) : ici la racine porte sur 4 + x", "93f3e11b46dc784d13a02783838305e6bda41a389ebf8b341c9454ff616183c0": "la dérivée seconde de √(4 + x) est négative", "e541dd20eaf8b24dce8c9316c050d3f178c51f41a2cb96f8ab198ec5cf307e30": "le terme d'ordre 2 est f''(0) x² / 2 : n'oublie pas le 2", "f91aabffcfc3e2f84a478e71fb054e9ac5173af67bfaef0757389035c5b9ab38": "l'ordre 2 ajoute un terme en x²"}, "points": [["1/3"], ["5/7"], ["6/5"], ["-2/5"], ["9/4"]], "variables": ["x"]}
📝 Ex CP1.7a : réponse enregistrée (expr).
WBM_ANSWER {"coarse_any": {"0": ["f4591760c99ea21ff7fa75a19202e7be2a16bcaca4a6f74e2834cd986ef1ee78"]}, "coarser": {"1": ["90b839c156b47216eea749a18aa036e434e1d3f4e8ceb4ce96fbe0efae2e965d"], "2": ["4fb6bdd066fd73513e51bd01a347607dc7cd2b7fbd0606b4edd1a5411704400

**CP1.8 — Gradient, dérivée directionnelle et point critique**

In [10]:
wbm.record("CP1.8a", GRAD_8, mistakes={"∂f/∂x dérive aussi −6xy, où y est une constante": [3, -12], "le gradient est [∂f/∂x, ∂f/∂y], dans cet ordre": [-12, 9], "attention aux signes : y vaut −1": [-3, 0]})
wbm.record("CP1.8b", Fraction(GRAD_8[0] * 20 + GRAD_8[1] * 21, 29), kind="fraction", simplified=True, mistakes={"une dérivée directionnelle se calcule avec un vecteur de norme 1": "-72"})
wbm.record("CP1.8c", math.hypot(*GRAD_8), decimals=3, mistakes={"la plus grande pente est la norme du gradient, pas son carré": 225, "la norme d'un vecteur n'est pas la somme des valeurs absolues de ses coordonnées": 21})
wbm.record("CP1.8d", [0, 0, 2, 2], mistakes={"range les points dans l'ordre des x croissants": [2, 2, 0, 0]})
wbm.record("CP1.8e", 3, choices=[1, 2, 3], mistakes={"calcule f_xx f_yy − (f_xy)² en ce point : son signe décide": 1})
wbm.record("CP1.8f", 2 ** 3 - 6 * 2 * 2 + 3 * 2 ** 2, mistakes={"c'est la valeur au premier point critique ; la question porte sur le second": 0, "attention au signe du terme −6xy": 44})

WBM_ANSWER {"decimals": 0, "element_hashes": ["00c36a97bbed43fc66b0607bcfcca073a878d1dad861b34b3cde3a691820f526", "cb765d78c5a8ad1e155e3dc04a5508f35050fe11509f559709b8c2ea4fa3a690"], "hash": "cf22e3c561d3bf42cb91697615b2a9596ace3be79b2059170c4d697b67c39dc9", "id": "CP1.8a", "integer": true, "kind": "array", "mistakes": {"62a9e4c764ec7dac02f12655437d4c95f2b1f955919cd47629495b113d0f84e0": "le gradient est [∂f/∂x, ∂f/∂y], dans cet ordre", "95c7afd42be9fa897938f30847912c79761e74c734a724484f44c54fbc1d2fc1": "attention aux signes : y vaut −1", "a167f22765336abc365cb452c3c5a12a6cdc1d6f9f74a428703a59ea4e906c8c": "∂f/∂x dérive aussi −6xy, où y est une constante"}, "shape": [2]}
📝 Ex CP1.8a : réponse enregistrée (array, 0 décimale).
WBM_ANSWER {"hash": "3d29528b85ded1d45f12b00d1ac96d01076b288183aa1c8b9c9c8faff9a5e6a7", "id": "CP1.8b", "kind": "fraction", "mistakes": {"87e53164007956bb5b06671f035884802ede7fece0e0223e9093395d0a940a99": "une dérivée directionnelle se calcule avec un vecteur de norm

**CP1.9 — Entropie, information mutuelle et KL**

In [11]:
wbm.record("CP1.9a", H(*P_9), decimals=3, mistakes={"les entropies sont en bits : log₂, pas ln": H(*P_9) * math.log(2)})
wbm.record("CP1.9b", CROSS_9, decimals=3, mistakes={"H(p, q) pondère par p les longueurs −log₂ q": -sum(q * math.log2(p) for p, q in zip(P_9, Q_9)), "c'est l'entropie de q ; la question demande la cross-entropy": H(*Q_9), "les entropies sont en bits : log₂, pas ln": CROSS_9 * math.log(2)})
wbm.record("CP1.9c", CROSS_9 - H(*P_9), decimals=3, mistakes={"KL(p ‖ q) pondère par p : l'ordre des deux lois compte": sum(q * math.log2(q / p) for p, q in zip(P_9, Q_9)), "une divergence KL est positive ou nulle": H(*P_9) - CROSS_9, "les entropies sont en bits : log₂, pas ln": (CROSS_9 - H(*P_9)) * math.log(2)})
wbm.record("CP1.9d", Fraction(35, 40), kind="fraction", simplified=True, mistakes={"P(question | F = 1) se lit dans la ligne F = 1, pas dans la colonne « question »": "7/9", "c'est la probabilité jointe P(question et F = 1)": "7/20"})
wbm.record("CP1.9e", H_Y_F9, decimals=2, mistakes={"H(Y | F) est une moyenne des entropies des lignes, pondérée par P(F), pas leur somme": H(0.875, 0.125) + H(1 / 6, 5 / 6), "c'est H(F) ; la question demande H(Y | F)": H(0.4, 0.6), "c'est H(F | Y) : la variable connue est à droite de la barre": 0.45 * H(35 / 45, 10 / 45) + 0.55 * H(5 / 55, 50 / 55), "les entropies sont en bits : log₂, pas ln": H_Y_F9 * math.log(2)})
wbm.record("CP1.9f", H(0.45, 0.55) - H_Y_F9, decimals=2, mistakes={"c'est H(Y | F), l'incertitude qui reste ; la question demande ce que F apprend sur Y": H_Y_F9, "garde au moins 3 décimales dans H(Y) et H(Y | F) avant de les soustraire": 0.38, "les entropies sont en bits : log₂, pas ln": (H(0.45, 0.55) - H_Y_F9) * math.log(2)})

WBM_ANSWER {"coarse_any": {"0": ["5de5ea8746d3f50b2933701297f846b50cac1dc2d555986ddd49d1c8e2cce433"]}, "coarser": {"1": ["25d26b77479a29937e0b12c2a9a4cfe77a82a2bff83c4ea9981a06b23b9e1749"]}, "decimals": 3, "hash": "161a1c52c5b6a3ebd653da61d2547bfdcf0536faa00b4375c0a76cb06ece3537", "hash_coarse": "f570f925fef87ee644619a11d67385668936b5b0d3893fc6a940e2e6618093e7", "id": "CP1.9a", "kind": "float", "magnitude": 0, "mistakes": {"363bd75e59a558a51909a11ae826d26dafc51338360cd10a4419e5707b14986e": "les entropies sont en bits : log₂, pas ln"}, "sign": 1}
📝 Ex CP1.9a : réponse enregistrée (float, 3 décimales).
WBM_ANSWER {"coarse_any": {"0": ["19719b3c3c30aae928a90350a751abfc987b66e210cde204868ba7ca80f9f57f"]}, "coarser": {"1": ["d95baa53495585b913663347397760db32928a28bcc796d39183882362023cef"]}, "decimals": 3, "hash": "726c6ad1837efcb7baaa174619ed22ac2c6bbc389e6535311327430d98ca4fa5", "hash_coarse": "a7e86cefed33856bf5563b16807a66ffb7befac10cf270eb8e5b2eec08b065a6", "id": "CP1.9b", "kind": "fl

**CP1.11 — Corriger une copie : cinq erreurs (les lettres seulement)**

In [12]:
wbm.record("CP1.11a", "ACDFG", mistakes={"il manque une des lignes fausses : il y en a exactement cinq": {'ACDF', 'ACDG', 'ACFG', 'ADFG', 'CDFG'}, "une des lignes citées est juste : relis-les une par une, avec la règle qu'elle utilise": {'ABCDFG', 'ACDEFG', 'ACDFGH'}, "une ligne juste est citée à la place d'une ligne fausse": {'ABCDF', 'ABCDG', 'ABCFG', 'ABDFG', 'ACDEF', 'ACDEG', 'ACDFH', 'ACDGH', 'ACEFG', 'ACFGH', 'ADEFG', 'ADFGH', 'BCDFG', 'CDEFG', 'CDFGH'}, "ce sont les trois lignes justes ; la question demande les cinq lignes fausses": 'BEH'})

WBM_ANSWER {"hash": "48fe8c8413cd9c33e7033d5e499f714bb8f474dea9c3ddcd780ea3cd85440a86", "id": "CP1.11a", "kind": "str", "mistakes": {"078eb36b4eec518cd3e9c3a465ddad778d2b4689198a452312a13e872fe6b721": "une ligne juste est citée à la place d'une ligne fausse", "10a6452ff428042d2ce1a98b75d112f5bbf7faecf6e2846dfaf90b65f257dff9": "une des lignes citées est juste : relis-les une par une, avec la règle qu'elle utilise", "1224689a0417e3c36ee7112655fdd95ecec9b29ff6c452ecaa0eeea98ced008e": "une ligne juste est citée à la place d'une ligne fausse", "2278dfd86482d449c3721a6ce1fe1b8ff4a7b6a5e0591a2a2d88ab5e35c675b8": "une ligne juste est citée à la place d'une ligne fausse", "40b26f6807b30e54cb65eb9727a8c506e419f2ddf570a737cffa02126503f025": "une ligne juste est citée à la place d'une ligne fausse", "54e4d0652dedc86302e39480b9cae7b19dc56c0031884242f892db66c9a58916": "une ligne juste est citée à la place d'une ligne fausse", "5d14ce61e9dc3667b2797056963a56e2f7c82c3572c8e3684d4c77715a022610": "une l

**CP1.12 — Problème de synthèse : estimer un paramètre de bout en bout**

In [13]:
wbm.record("CP1.12a", 2 * math.exp(-2) / (1 - math.exp(-2)), decimals=3, mistakes={"le journal ne garde que les jours qui ont au moins un signalement : relis le dénominateur de la loi de l'énoncé": 2 * math.exp(-2)})
wbm.record("CP1.12b", "32/lam - 20/(1 - exp(-lam))", kind="expr", variables=("lam",), mistakes={"dérive aussi le terme −20 ln(1 − e^(−λ))": "32/lam - 20", "la dérivée de ln(1 − e^(−λ)) est e^(−λ) / (1 − e^(−λ)) : attention au signe": "32/lam - 20 + 20*exp(-lam)/(1 - exp(-lam))", "la log-vraisemblance somme les 20 jours": "1.6/lam - 1/(1 - exp(-lam))"})
wbm.record("CP1.12c", NEWTON_12[1], decimals=3, mistakes={"un pas de Newton retranche g(λ)/g'(λ)": 1.6 + (1.6 - XBAR_12 * (1 - math.exp(-1.6))) / (1 - XBAR_12 * math.exp(-1.6)), "la dérivée de −1,6 (1 − e^(−λ)) est −1,6 e^(−λ)": 1.6 - (1.6 - XBAR_12 * (1 - math.exp(-1.6))) / (1 + XBAR_12 * math.exp(-1.6))})
wbm.record("CP1.12d", NEWTON_12[2], decimals=3, mistakes={"le deuxième pas part de λ₁, avec g et g' recalculés en λ₁": NEWTON_12[1]})
wbm.record("CP1.12e", LAM_12, decimals=3, mistakes={"la moyenne des jours du journal ne compte pas les jours sans signalement : résous g(λ) = 0": XBAR_12, "continue les pas jusqu'à ce que les trois décimales ne bougent plus": NEWTON_12[2]})
wbm.record("CP1.12f", True, mistakes={"calcule ℓ'(1) avec ton expression de b)": False})
wbm.record("CP1.12g", math.exp(-LAM_12), decimals=3, mistakes={"la proportion de jours sans signalement se calcule avec le paramètre estimé, pas avec la moyenne observée": math.exp(-XBAR_12)})
wbm.record("CP1.12h", [LAM_12 - 1.96 * 0.28, LAM_12 + 1.96 * 0.28], decimals=2, mistakes={"l'intervalle est centré sur l'estimation de λ, pas sur la moyenne observée": [XBAR_12 - 1.96 * 0.28, XBAR_12 + 1.96 * 0.28], "un intervalle à 95 % s'étend de combien d'écarts-types de part et d'autre ?": [LAM_12 - 0.28, LAM_12 + 0.28], "0,28 est déjà l'écart-type de l'estimation : ne le divise pas par √n": [LAM_12 - 1.96 * 0.28 / math.sqrt(20), LAM_12 + 1.96 * 0.28 / math.sqrt(20)]})

WBM_ANSWER {"coarse_any": {"1": ["f34d4a59de3b6ffa8c6c4cde854c63ec45e1c115ce18e972a3fc97b1c97d35dd"]}, "decimals": 3, "hash": "5836f50537beda76fe177aba47dcfca7b3d5903988283c43a056c8b1870023c3", "hash_coarse": "8dd3c854ce6727a47fa189cbd58f26dcd6dfdd1e745c2b09ffcfe43782eb51cc", "id": "CP1.12a", "kind": "float", "magnitude": -1, "mistakes": {"20e4fd00adaea955f19ad1d493e5037f7fa5acc3170f3f34b902f0cccc256e9f": "le journal ne garde que les jours qui ont au moins un signalement : relis le dénominateur de la loi de l'énoncé"}, "sign": 1}
📝 Ex CP1.12a : réponse enregistrée (float, 3 décimales).
WBM_ANSWER {"hash": "de4091b6811317ef915595d881e3844939336c96141b61bd0d33885c797a254d", "id": "CP1.12b", "kind": "expr", "mistakes": {"2a7614cfc37e1739afd188b0e5eb1a94aacc275268e5d2acfd455abb69d75379": "la log-vraisemblance somme les 20 jours", "4ad7ae7e48a8f339698ca09e0cfb2b75f1d4f297f140d6ecb3d225bacf49e23c": "la dérivée de ln(1 − e^(−λ)) est e^(−λ) / (1 − e^(−λ)) : attention au signe", "d1c7e5e1bd213c